# Bangalore Tech Salary Decoder
# Live Project - Unlox Academy

# Name: JEEVIKA S

## Task 1 - Load and inspect

In [1]:
# loading the csv and checking basic stuff
import pandas as pd
import numpy as np

df = pd.read_csv('bangalore_tech_salaries.csv')
print(df.shape)
df.info()


(1015, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB


In [2]:
# checking nulls and duplicates
print(df.isnull().sum())
print("duplicates:", df.duplicated().sum())


Employee_ID         0
Role                0
years_exp           0
Current_CTC         0
Previous_CTC      200
Company             0
company_TYPE        0
Skills             27
Location           20
Education_Tier      0
Joining_Year        0
Work_Mode           0
dtype: int64
duplicates: 15


Summary:
- 1015 rows, 12 columns
- Current_CTC and Previous_CTC are strings not numbers, need to fix
- Previous_CTC has a lot of nulls (~20%) - probably freshers
- Skills column also has some nulls
- 15 duplicate rows
- Role, company_TYPE, Education_Tier all have messy variants

## Task 2 - Cleaning

In [3]:
# rename columns, the Role column has a trailing space in the header
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={
    'Employee_ID':'employee_id','Role':'role','years_exp':'years_exp',
    'Current_CTC':'current_ctc','Previous_CTC':'previous_ctc','Company':'company',
    'company_TYPE':'company_type','Skills':'skills','Location':'location',
    'Education_Tier':'education_tier','Joining_Year':'joining_year','Work_Mode':'work_mode'
})
df['role'] = df['role'].str.strip()
df = df.drop_duplicates()
print(df.shape)


(1000, 12)


In [4]:
# checked df['role'].unique() first, then made this mapping by hand
role_map = {
    'DS':'Data Scientist','Data Scientist':'Data Scientist','Data Science Engineer':'Data Scientist',
    'DevOps':'DevOps Engineer','DevOps Engineer':'DevOps Engineer','Infra Engineer':'DevOps Engineer',
    'SDE Backend':'SDE Backend','SDE-Backend':'SDE Backend','Backend Dev':'SDE Backend',
    'Backend Developer':'SDE Backend','Backend Engineer':'SDE Backend','BE':'SDE Backend',
    'SDE Frontend':'SDE Frontend','SDE-Frontend':'SDE Frontend','Frontend Dev':'SDE Frontend',
    'Frontend Engineer':'SDE Frontend','Frontend Developer':'SDE Frontend','FE':'SDE Frontend',
    'Product Lead':'Product Manager','Product Manager':'Product Manager','Sr PM':'Product Manager','PM':'Product Manager',
    'Site Reliability Engineer':'Site Reliability Engineer','SRE':'Site Reliability Engineer',
    'DA':'Data Analyst','Data Analyst':'Data Analyst',
    'UI/UX':'UI/UX Designer','UI Designer':'UI/UX Designer','Designer':'UI/UX Designer',
    'Product Designer':'UI/UX Designer','UX Designer':'UI/UX Designer',
    'ML Engineer':'ML Engineer',
    'Full Stack Engineer':'SDE Full-Stack','FullStack':'SDE Full-Stack','SDE FS':'SDE Full-Stack',
    'Fullstack Dev':'SDE Full-Stack','SDE Full-Stack':'SDE Full-Stack',
    'BA':'Business Analyst','Business Analyst':'Business Analyst','BI Analyst':'Business Analyst',
    'Business Systems Analyst':'Business Analyst','Analytics Engineer':'Business Analyst',
}
df['role_clean'] = df['role'].map(role_map)
print(df['role_clean'].isnull().sum())
print(df['role_clean'].value_counts())


0
role_clean
Business Analyst             158
SDE Backend                  114
UI/UX Designer               113
Product Manager              112
SDE Frontend                 105
SDE Full-Stack                96
Data Scientist                94
DevOps Engineer               66
Site Reliability Engineer     61
Data Analyst                  50
ML Engineer                   31
Name: count, dtype: int64


In [5]:
# CTC is in 4 formats - '15.5 LPA', '15.5', '15,50,000', '₹15.5 LPA'
# the tricky one is '15,50,000' -> that's indian number format, means 15.5 LPA not 1.5 million
def parse_ctc(x):
    if pd.isnull(x):
        return np.nan
    x = str(x).strip().replace('₹','').replace('LPA','').strip().replace(',','')
    try:
        val = float(x)
    except:
        return np.nan
    if val > 1000:   # this means it was actual rupees not LPA, so convert
        val = val/100000
    return round(val,2)

df['ctc_lpa'] = df['current_ctc'].apply(parse_ctc)
df['prev_ctc_lpa'] = df['previous_ctc'].apply(parse_ctc)  # leaving NaN as NaN, not filling with 0
df = df.dropna(subset=['ctc_lpa'])
print(df[['current_ctc','ctc_lpa']].head())


  current_ctc  ctc_lpa
0        49.4     49.4
1         9.7      9.7
2   3,360,000     33.6
3    41.4 LPA     41.4
4   3,640,000     36.4


In [6]:
# company type has case issues - unicorn vs UNICORN etc
df['company_type_clean'] = df['company_type'].str.strip().str.lower().map({
    'unicorn':'Unicorn','mnc':'MNC','mid-size':'Mid-size','midsize':'Mid-size',
    'early-stage':'Early-stage','earlystage':'Early-stage'
})
print(df['company_type_clean'].value_counts())


company_type_clean
MNC            342
Mid-size       257
Unicorn        207
Early-stage    194
Name: count, dtype: int64


In [7]:
# education tier - Tier 1, T1, 1, Tier-1 all same thing
def fix_tier(x):
    if pd.isnull(x):
        return np.nan
    x = str(x).lower().replace('-','').replace(' ','')
    if '1' in x: return 'Tier 1'
    if '2' in x: return 'Tier 2'
    if '3' in x: return 'Tier 3'
    return np.nan

df['education_tier_clean'] = df['education_tier'].apply(fix_tier)
print(df['education_tier_clean'].value_counts())


education_tier_clean
Tier 2    500
Tier 3    306
Tier 1    194
Name: count, dtype: int64


In [8]:
# final check after cleaning
print(df.dtypes)
print(df['role_clean'].value_counts())
print(df['company_type_clean'].value_counts())
print(df['education_tier_clean'].value_counts())


employee_id              object
role                     object
years_exp                 int64
current_ctc              object
previous_ctc             object
company                  object
company_type             object
skills                   object
location                 object
education_tier           object
joining_year              int64
work_mode                object
role_clean               object
ctc_lpa                 float64
prev_ctc_lpa            float64
company_type_clean       object
education_tier_clean     object
dtype: object
role_clean
Business Analyst             158
SDE Backend                  114
UI/UX Designer               113
Product Manager              112
SDE Frontend                 105
SDE Full-Stack                96
Data Scientist                94
DevOps Engineer               66
Site Reliability Engineer     61
Data Analyst                  50
ML Engineer                   31
Name: count, dtype: int64
company_type_clean
MNC            342
Mid-

## Task 3 - Business questions

### Q1 - CTC by role

In [9]:
role_stats = df.groupby('role_clean')['ctc_lpa'].agg(['median','mean','min','max'])
role_stats = role_stats.sort_values('median', ascending=False).round(2)
print(role_stats)


                           median   mean   min   max
role_clean                                          
Product Manager             31.30  34.13  10.8  80.1
ML Engineer                 27.50  30.34  10.0  64.5
Data Scientist              24.15  26.67  10.9  75.6
Site Reliability Engineer   23.30  23.77   8.8  55.0
SDE Full-Stack              22.35  25.41   8.9  71.7
SDE Frontend                21.20  22.19   6.7  84.4
SDE Backend                 21.10  23.28   7.9  55.1
DevOps Engineer             19.85  22.81   9.2  60.3
UI/UX Designer              18.90  20.97   6.2  63.3
Business Analyst            18.80  20.84   5.2  52.7
Data Analyst                16.30  17.15   5.6  43.4


Insight: Product Manager has the highest median CTC (~31.3 LPA) and Data Analyst the lowest
(~16.3 LPA), almost 2x difference. Mean is higher than median for PM and ML Engineer which means
some really high earners are pulling the average up in those roles.

### Q2 - SDE Backend experience curve

In [10]:
sde = df[df['role_clean']=='SDE Backend'].copy()
sde['exp_band'] = pd.cut(sde['years_exp'], bins=[-1,1,3,5,100], labels=['0-1','2-3','4-5','6+'])
curve = sde.groupby('exp_band')['ctc_lpa'].median()
print(curve)
print(curve.pct_change()*100)


exp_band
0-1    11.65
2-3    20.00
4-5    25.85
6+     40.40
Name: ctc_lpa, dtype: float64
exp_band
0-1          NaN
2-3    71.673820
4-5    29.250000
6+     56.286267
Name: ctc_lpa, dtype: float64


/tmp/ipykernel_3551/3498344642.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  curve = sde.groupby('exp_band')['ctc_lpa'].median()


Insight: median CTC goes from ~11.6 LPA at 0-1 yrs to ~40.4 LPA at 6+ yrs, about 3.5x growth.
Biggest jump is between 0-1 and 2-3 years (+72%), so the early years matter the most for pay growth.

### Q3 - skill premium

In [11]:
sde_roles = ['SDE Backend','SDE Frontend','SDE Full-Stack']
sdes = df[df['role_clean'].isin(sde_roles)].copy()
sdes['skills'] = sdes['skills'].fillna('')

for skill in ['AWS','ML','System Design','Kubernetes']:
    has = sdes[sdes['skills'].str.contains(skill, case=False)]['ctc_lpa'].median()
    no = sdes[~sdes['skills'].str.contains(skill, case=False)]['ctc_lpa'].median()
    print(skill, "with:", has, "without:", no, "premium%:", round((has/no-1)*100,1))


AWS with: 20.95 without: 21.9 premium%: -4.3
ML with: 23.95 without: 21.3 premium%: 12.4
System Design with: 25.3 without: 21.0 premium%: 20.5
Kubernetes with: 23.2 without: 21.5 premium%: 7.9


Insight: System Design gives the biggest boost, about +20% median CTC. AWS barely makes a
difference (slightly negative even), probably because almost everyone already has it so it's not
special anymore.

### Q4 - company type premium

In [12]:
q4 = df[df['role_clean']=='SDE Backend'].groupby('company_type_clean')['ctc_lpa'].median()
q4 = q4.sort_values(ascending=False)
print(q4)
print((q4/q4['Unicorn'] - 1)*100)


company_type_clean
Unicorn        27.35
MNC            20.45
Mid-size       19.50
Early-stage    18.60
Name: ctc_lpa, dtype: float64
company_type_clean
Unicorn         0.000000
MNC           -25.228519
Mid-size      -28.702011
Early-stage   -31.992687
Name: ctc_lpa, dtype: float64


Insight: Unicorns pay SDE Backend the most (~27.4 LPA median). MNC is about 25% less, and
Early-stage is about 32% less than Unicorn for the same role.

### Q5 - underpaid people

In [13]:
df['exp_band'] = pd.cut(df['years_exp'], bins=[-1,1,3,5,100], labels=['0-1','2-3','4-5','6+'])

grp_cols = ['role_clean','company_type_clean','exp_band']
df['group_median'] = df.groupby(grp_cols)['ctc_lpa'].transform('median')
df['group_size'] = df.groupby(grp_cols)['ctc_lpa'].transform('size')
df['gap'] = df['ctc_lpa'] - df['group_median']

reliable = df[df['group_size'] >= 10]   # only trust groups with 10+ people
top10 = reliable.sort_values('gap').head(10)[['employee_id','role_clean','company_type_clean','years_exp','ctc_lpa','group_median','gap']]
print(top10)


/tmp/ipykernel_3551/3962569165.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['group_median'] = df.groupby(grp_cols)['ctc_lpa'].transform('median')
/tmp/ipykernel_3551/3962569165.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['group_size'] = df.groupby(grp_cols)['ctc_lpa'].transform('size')


    employee_id        role_clean company_type_clean  years_exp  ctc_lpa  \
349     BLR0954  Business Analyst                MNC          6     25.4   
904     BLR0925       SDE Backend                MNC          4     19.4   
250     BLR0072       SDE Backend                MNC          4     19.5   
530     BLR0441  Business Analyst            Unicorn          4     23.1   
428     BLR0593  Business Analyst           Mid-size          6     24.1   
728     BLR0301   Product Manager                MNC          2     24.7   
249     BLR0733  Business Analyst                MNC          7     31.1   
753     BLR0611   Product Manager                MNC          2     25.1   
688     BLR0203       SDE Backend                MNC          4     21.3   
711     BLR0451      SDE Frontend                MNC          4     19.5   

     group_median    gap  
349         37.65 -12.25  
904         27.30  -7.90  
250         27.30  -7.80  
530         30.35  -7.25  
428         30.90  -6.80  
7

Insight: the most underpaid person is about 12.3 LPA below their group's median, which is huge.
Business Analyst and SDE Backend show up multiple times in this list so those roles seem to have
the most unfair pay spread within the same experience/company-type group.

## Task 4 - Final report

In [15]:
# building the printed report
print("="*60)
print(" BANGALORE TECH SALARY DECODER")
print(" Built by Jeevika | Unlox Academy")
print("="*60)
print(f" Dataset: {df.shape[0]} bengaluru tech employees (cleaned)")
print()
print(" ---- MEDIAN CTC BY ROLE ----")
for role, row in role_stats.iterrows():
    print(f" {role:<25}{row['median']:>6.1f} LPA")
print()
print(" ---- SDE BACKEND BY EXPERIENCE ----")
prev = None
for band, val in curve.items():
    if prev is None:
        print(f" {band:<8}: {val:.1f} LPA")
    else:
        g = (val/prev - 1)*100
        print(f" {band:<8}: {val:.1f} LPA (+{g:.0f}%)")
    prev = val
print()
print(" ---- COMPANY TYPE PREMIUM (SDE Backend) ----")
for ctype, val in q4.items():
    print(f" {ctype:<12}{val:>6.1f} LPA")
print()
print(" ---- TOP 5 UNDERPAID ----")
for _, r in top10.head(5).iterrows():
    print(f" {r['employee_id']} {r['role_clean']} {r['company_type_clean']} {r['years_exp']}yrs gap:{r['gap']:.1f}")
print("="*60)


 BANGALORE TECH SALARY DECODER
 Built by Jeevika | Unlox Academy
 Dataset: 1000 bengaluru tech employees (cleaned)

 ---- MEDIAN CTC BY ROLE ----
 Product Manager            31.3 LPA
 ML Engineer                27.5 LPA
 Data Scientist             24.1 LPA
 Site Reliability Engineer  23.3 LPA
 SDE Full-Stack             22.4 LPA
 SDE Frontend               21.2 LPA
 SDE Backend                21.1 LPA
 DevOps Engineer            19.9 LPA
 UI/UX Designer             18.9 LPA
 Business Analyst           18.8 LPA
 Data Analyst               16.3 LPA

 ---- SDE BACKEND BY EXPERIENCE ----
 0-1     : 11.6 LPA
 2-3     : 20.0 LPA (+72%)
 4-5     : 25.9 LPA (+29%)
 6+      : 40.4 LPA (+56%)

 ---- COMPANY TYPE PREMIUM (SDE Backend) ----
 Unicorn       27.4 LPA
 MNC           20.5 LPA
 Mid-size      19.5 LPA
 Early-stage   18.6 LPA

 ---- TOP 5 UNDERPAID ----
 BLR0954 Business Analyst MNC 6yrs gap:-12.2
 BLR0925 SDE Backend MNC 4yrs gap:-7.9
 BLR0072 SDE Backend MNC 4yrs gap:-7.8
 BLR0441 Busin

## Task 5 - Three insights

1. Product Managers earn almost double what Data Analysts earn at the median (31.3 vs 16.3 LPA)
- if you're choosing between similar offers, PM track pays a lot more long term.

2. Unicorns pay SDE Backend about 25% more than MNCs and 32% more than Early-stage at the same
experience level - worth weighing this heavily when comparing offers.

3. System Design adds about 20% to median CTC for SDEs, way more than AWS which barely matters
anymore - worth prioritizing System Design prep over another cloud cert.

## Task 6 - Review
- notebook runs start to end, no errors
- comments added in each cell
- AI-assisted: used Claude to help figure out the CTC parsing logic and structure the report, wrote the insights and comments myself after checking the output